# 12 — Error analysis: where does the final model fail, and why?

**Purpose.** Every model in this project lands at about 0.82–0.825 macro-F1 and Ankylosing Spondylitis (AS) stays at F1 ≈ 0.66 no matter what we change. This notebook asks *why*, with two competing explanations:

* **Hypothesis M (missing data):** the ceiling exists because many patients lack the tests that would separate the diseases (RF, Anti-CCP, HLA-B27, … are missing for 9–43% of patients), and imputation cannot recreate information that was never measured.
* **Hypothesis O (class overlap):** the diseases genuinely overlap in the measured values, so even a fully-measured patient would often be misclassified.

The two predict different things: under M, error rates climb steeply with the number of missing tests and fall sharply for patients with nearly complete data; under O, error rates are roughly flat.

**Reads:** `../data/raw/dataset.xlsx` (training pool only), `../results/tune_oof.npz`, `../results/tune_summary.csv`, `../results/fs_importance_stability.csv` · **Writes:** `../results/err_*.csv`

**Important: this notebook never touches the test set.** It uses the *out-of-fold* predictions of the final model (XGBoost with settings tuned inside each fold, from notebook 07): every prediction was made by a model that never saw that patient.

**What is exploratory, and what is not.** Nothing here selects a model or a feature, so no decision depends on it. The findings are *descriptive*; the interpretation rule below is fixed in advance so the conclusion cannot be reshaped after the numbers are seen.

**Pre-specified interpretation rule**

| Evidence | Meaning |
|---|---|
| Each extra missing test multiplies the odds of an error by a factor whose 95% interval excludes 1 **and** accuracy for patients with ≤ 1 missing test exceeds overall accuracy by **≥ 0.05** | Hypothesis M is supported: missing data is a main cause of the ceiling |
| Interval includes 1, **or** accuracy for ≤ 1 missing test is within **0.02** of overall | Hypothesis O is supported: the ceiling is class overlap, not missingness |
| Anything in between | Mixed: both contribute |

**Caveats.** Missing values here look randomly inserted (notebook 01b), so a test being missing is not itself informative; the error increase when a test is missing therefore measures how much *that test's information* is worth. Pair-level analyses (section 5) choose the "key markers" from the same patients they describe; that is acceptable for description but means no p-value there is a confirmatory test.

## 0. Setup

**What this does:** imports libraries and the shared helpers.

In [1]:
import sys, os, json, warnings
sys.path.append('..')

import numpy as np
import pandas as pd
import scipy
from scipy.stats import mannwhitneyu, fisher_exact, chi2_contingency, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

from src.preprocessing.cv_pipeline import load_cv_pool
from src.evaluation.metrics import macro_f1_fast, per_class_prf, holm

print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scipy 1.18.1


## 1. Load the training pool and the final model's out-of-fold predictions

**What this does:** loads the 9,668 training patients (with their gaps still present), loads the out-of-fold probabilities of the final model, and checks they belong together.

**Checks (all must pass):**
* the patients are the same, in the same order, as notebook 02's training set;
* the labels stored with the predictions equal the labels of the pool;
* the pooled macro-F1 recomputed here equals the value stored by notebook 07 (`tune_summary.csv`), proving these are the predictions that notebook scored.

In [2]:
pool = load_cv_pool("../data/raw/dataset.xlsx")
assert pool.index.equals(pd.read_csv('../data/processed/y_train.csv', index_col=0).index)
y_global = LabelEncoder().fit_transform(pool['Disease'])
classes = sorted(pool['Disease'].unique()); K = len(classes)

z = np.load('../results/tune_oof.npz', allow_pickle=True)
P = z['XGBoost__tuned'].astype(float)
assert (z['y'] == y_global).all() and P.shape == (len(pool), K)
pred = P.argmax(1); y = y_global
err = (pred != y).astype(int)

t07 = pd.read_csv('../results/tune_summary.csv').set_index(['model', 'variant'])['oof_macro_f1']
mine = macro_f1_fast(y, pred, K)
print(f"pooled OOF macro-F1: {mine:.4f} | notebook 07: {t07[('XGBoost', 'tuned')]:.4f} | accuracy: {1 - err.mean():.4f}")
assert abs(mine - t07[('XGBoost', 'tuned')]) < 1e-4
print("alignment OK")

pooled OOF macro-F1: 0.8254 | notebook 07: 0.8254 | accuracy: 0.8390
alignment OK


## 2. The missingness table

**What this does:** builds, for every patient, which of the 12 measured columns are missing and how many (`n_missing`). The distribution of `n_missing` is printed.

**Check:** each column's missing count in the training pool should be about 80% of its whole-dataset count from notebook 01b (an `ok` column and a warning line are printed), and no patient has all 12 missing.

In [3]:
cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']
miss = pool[cols].isna()
n_missing = miss.sum(axis=1).to_numpy()
assert miss.shape[1] == 12 and n_missing.max() < 12
# whole-dataset counts from notebook 01b / 02; an 80/20 split should keep ~80% of each in the training pool
global_missing = {'ESR': 1088, 'CRP': 2417, 'RF': 1329, 'Anti-CCP': 3263, 'HLA-B27': 1934, 'ANA': 3746,
                  'Anti-Ro': 2900, 'Anti-La': 3021, 'Anti-dsDNA': 4713, 'Anti-Sm': 5197, 'C3': 1692, 'C4': 2054}
chk = pd.DataFrame({'train_pool': miss.sum(), 'expected_about': [round(0.8 * global_missing[c]) for c in cols]})
chk['ok'] = (chk['train_pool'] - chk['expected_about']).abs() <= 0.04 * pd.Series(global_missing)
print("missing per column (training pool) vs ~80% of the whole-dataset count:")
print(chk.to_string())
if not chk['ok'].all():
    print("WARNING: some counts are off by more than 4% of the whole-dataset count - check that this is the real dataset.")
dist = pd.Series(n_missing).value_counts().sort_index()
print("\npatients by number of missing tests:")
print(pd.DataFrame({'patients': dist, 'share': (dist / len(pool)).round(3)}).T.to_string())

missing per column (training pool) vs ~80% of the whole-dataset count:
            train_pool  expected_about    ok
ESR                876             870  True
CRP               1925            1934  True
RF                1067            1063  True
Anti-CCP          2591            2610  True
HLA-B27           1555            1547  True
ANA               3024            2997  True
Anti-Ro           2317            2320  True
Anti-La           2390            2417  True
Anti-dsDNA        3757            3770  True
Anti-Sm           4171            4158  True
C3                1347            1354  True
C4                1626            1643  True

patients by number of missing tests:
                0         1         2         3         4        5        6       7       8
patients  369.000  1441.000  2573.000  2535.000  1688.000  746.000  251.000  54.000  11.000
share       0.038     0.149     0.266     0.262     0.175    0.077    0.026   0.006   0.001


## 3. Does the error rate climb with the number of missing tests?

**3a — Accuracy by number of missing tests.** Patients are grouped by how many of the 12 tests are missing (0–1, 2, 3, 4, 5 or more). For each group: size, accuracy, macro-F1 and a 95% Wilson interval for the accuracy. Macro-F1 of small groups is noisy (some classes have few patients) — `n_ReA` shows how many Reactive Arthritis patients each group has.

**3b — Effect size.** A logistic regression of "prediction is wrong" on `n_missing`, adjusting for the patient's true class (because classes differ in how often their tests are missing and in how hard they are). The coefficient is turned into an **odds ratio per extra missing test**, with a 95% interval from 1,000 bootstrap resamples of patients. An odds ratio of 1.0 means missingness does not matter; 1.15 would mean each missing test raises the odds of error by 15%.

In [4]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (c - h, c + h)

bins = pd.cut(n_missing, [-1, 1, 2, 3, 4, 20], labels=['0-1', '2', '3', '4', '5+'])
rows = []
for b in bins.categories:
    m = (bins == b)
    lo, hi = wilson(int((1 - err[m]).sum()), int(m.sum()))
    rows.append({'n_missing': b, 'patients': int(m.sum()), 'n_ReA': int((y[m] == classes.index('Reactive Arthritis')).sum()),
                 'accuracy': 1 - err[m].mean(), 'acc_ci_low': lo, 'acc_ci_high': hi,
                 'macro_f1': macro_f1_fast(y[m], pred[m], K)})
by_missing = pd.DataFrame(rows)
print(by_missing.round(4).to_string(index=False))
acc_all = 1 - err.mean()
acc_low = float(by_missing.loc[by_missing.n_missing == '0-1', 'accuracy'].iloc[0])
print(f"\noverall accuracy {acc_all:.4f} | accuracy with <= 1 missing test {acc_low:.4f} | difference {acc_low - acc_all:+.4f}")

n_missing  patients  n_ReA  accuracy  acc_ci_low  acc_ci_high  macro_f1
      0-1      1810     71    0.8967      0.8818       0.9099    0.8892
        2      2573    102    0.8651      0.8514       0.8778    0.8511
        3      2535    103    0.8300      0.8149       0.8441    0.8135
        4      1688     85    0.8039      0.7843       0.8222    0.7898
       5+      1062     52    0.7542      0.7275       0.7792    0.7429

overall accuracy 0.8390 | accuracy with <= 1 missing test 0.8967 | difference +0.0577


In [5]:
D = np.eye(K)[y][:, 1:]
def odds_ratio(idx):
    X = np.column_stack([n_missing[idx], D[idx]])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m = LogisticRegression(C=1e6, max_iter=2000).fit(X, err[idx])
    return float(np.exp(m.coef_[0][0]))
rng = np.random.default_rng(42)
ors = [odds_ratio(rng.integers(0, len(y), len(y))) for _ in range(1000)]
or_hat = odds_ratio(np.arange(len(y)))
or_lo, or_hi = np.percentile(ors, [2.5, 97.5])
print(f"odds ratio of an error per extra missing test (adjusted for true class): {or_hat:.3f}  95% CI [{or_lo:.3f}, {or_hi:.3f}]")

odds ratio of an error per extra missing test (adjusted for true class): 1.315  95% CI [1.261, 1.374]


### 3c — Applying the pre-specified rule

**What this does:** applies the rule from the top of the notebook mechanically to the two numbers above and prints which hypothesis the evidence supports.

In [6]:
gap = acc_low - acc_all
if or_lo > 1 and gap >= 0.05:
    verdict = "HYPOTHESIS M SUPPORTED: missing data is a main cause of the ceiling"
elif or_lo <= 1 or gap <= 0.02:
    verdict = "HYPOTHESIS O SUPPORTED: the ceiling is class overlap, not missingness"
else:
    verdict = "MIXED: both missingness and class overlap contribute"
print(f"odds-ratio CI excludes 1: {or_lo > 1} | accuracy gain for <=1 missing test: {gap:+.4f}")
print("VERDICT:", verdict)

odds-ratio CI excludes 1: True | accuracy gain for <=1 missing test: +0.0577
VERDICT: HYPOTHESIS M SUPPORTED: missing data is a main cause of the ceiling


## 4. Which missing test hurts most? (and a second check on feature importance)

**What this does:** for each of the 12 tests, compares the error rate of patients **where that test is missing** with patients **where it is measured**. Because gaps are inserted at random, the difference tells us how much information the model loses without that test. The two-proportion χ² p-values are Holm-corrected across the 12 tests.

**Cross-check with notebook 06:** the ranking of tests by `error_diff` is compared (Spearman correlation) with the permutation-importance ranking from notebook 06. These are two independent ways of measuring "how much does the model rely on this test"; agreement is reassuring, disagreement would be worth understanding. (Tests that notebook 06 found irrelevant — Anti-dsDNA, Anti-Sm — should show an error difference near 0.)

In [7]:
rows = []
for c in cols:
    m = miss[c].to_numpy()
    tab = np.array([[err[m].sum(), (1 - err[m]).sum()], [err[~m].sum(), (1 - err[~m]).sum()]])
    chi2, p, _, _ = chi2_contingency(tab, correction=False)
    rows.append({'test': c, 'n_missing': int(m.sum()), 'err_when_missing': err[m].mean(), 'err_when_measured': err[~m].mean(),
                 'error_diff': err[m].mean() - err[~m].mean(), 'p_raw': p})
perm = pd.DataFrame(rows)
perm['p_holm'] = holm(perm['p_raw'])
perm = perm.sort_values('error_diff', ascending=False).reset_index(drop=True)
print(perm.drop(columns='p_raw').round(4).to_string(index=False))

imp_path = '../results/fs_importance_stability.csv'
if os.path.exists(imp_path):
    imp = pd.read_csv(imp_path)
    imp = imp[(imp.model == 'XGBoost') & imp.feature.isin(cols)].set_index('feature')['mean_importance']
    merged = perm.set_index('test').join(imp.rename('notebook06_importance')).dropna()
    rho, p_rho = spearmanr(merged['error_diff'], merged['notebook06_importance'])
    print(f"\nSpearman correlation, error_diff vs notebook-06 permutation importance (n={len(merged)}): rho={rho:.2f}, p={p_rho:.3f}")
else:
    print("fs_importance_stability.csv not found - skipping the cross-check")

      test  n_missing  err_when_missing  err_when_measured  error_diff  p_holm
       ESR        876            0.2808             0.1491      0.1317  0.0000
        RF       1067            0.2718             0.1473      0.1245  0.0000
  Anti-CCP       2591            0.2088             0.1436      0.0652  0.0000
       CRP       1925            0.1917             0.1534      0.0383  0.0004
        C3       1347            0.1849             0.1572      0.0277  0.0607
        C4       1626            0.1814             0.1569      0.0245  0.0607
   Anti-Ro       2317            0.1795             0.1552      0.0243  0.0438
   Anti-La       2390            0.1778             0.1555      0.0223  0.0607
       ANA       3024            0.1763             0.1541      0.0221  0.0438
   HLA-B27       1555            0.1788             0.1576      0.0211  0.1135
Anti-dsDNA       3757            0.1594             0.1621     -0.0026  0.8864
   Anti-Sm       4171            0.1578             

## 5. The confusing disease pairs: do errors sit where the key tests are missing?

**What this does**
1. Finds the 4 most frequent confused pairs (counting A→B and B→A together) from the out-of-fold confusion matrix.
2. For each pair, takes patients whose true class is one of the two and finds the **3 most discriminating tests** between them (area under the ROC curve of each test, on measured values only; 0.5 = no separation).
3. Counts, per patient, how many of those 3 key tests are missing (0, 1, 2–3) and reports the share of patients **predicted as the other class of the pair**.
4. Fisher's exact test compares "all key tests measured" with "at least one missing" (Holm-corrected over the 4 pairs).

**How to read it:** if the confusion rate is the same whether the key tests are present or missing, the pair is confused *even when the information is there* (overlap). If it jumps when a key test is missing, that pair's errors are partly a missing-data problem. Exploratory (see caveats at the top).

In [8]:
cm = np.bincount(y * K + pred, minlength=K * K).reshape(K, K)
pairs = sorted({tuple(sorted((i, j))) for i in range(K) for j in range(K) if i != j},
               key=lambda ij: -(cm[ij[0], ij[1]] + cm[ij[1], ij[0]]))[:4]
out, pair_rows = [], []
for a, b in pairs:
    sel = np.where((y == a) | (y == b))[0]
    conf = np.where(y[sel] == a, pred[sel] == b, pred[sel] == a).astype(int)
    auc = {}
    for c in cols:
        xa = pool[c].to_numpy(float)[sel][y[sel] == a]; xb = pool[c].to_numpy(float)[sel][y[sel] == b]
        xa, xb = xa[~np.isnan(xa)], xb[~np.isnan(xb)]
        auc[c] = mannwhitneyu(xa, xb).statistic / (len(xa) * len(xb))
    key = sorted(cols, key=lambda c: -abs(auc[c] - 0.5))[:3]
    nk = miss[key].to_numpy()[sel].sum(1)
    print(f"\n=== {classes[a]}  vs  {classes[b]}  (confused {cm[a, b] + cm[b, a]} times) ===")
    print("key tests:", ", ".join(f"{c} (AUC {auc[c]:.2f})" for c in key))
    for lbl, m in (('all 3 measured', nk == 0), ('1 missing', nk == 1), ('2-3 missing', nk >= 2)):
        print(f"  {lbl:<15} patients={int(m.sum()):>5}  confused={conf[m].mean() if m.sum() else float('nan'):.3f}")
    any_m = nk >= 1
    tab = np.array([[conf[any_m].sum(), (1 - conf[any_m]).sum()], [conf[~any_m].sum(), (1 - conf[~any_m]).sum()]])
    odds, p = fisher_exact(tab)
    pair_rows.append({'pair': f"{classes[a]} vs {classes[b]}", 'key_tests': ', '.join(key),
                      'confused_all_measured': conf[~any_m].mean(), 'confused_any_missing': conf[any_m].mean(),
                      'odds_ratio': odds, 'p_raw': p})
pair_df = pd.DataFrame(pair_rows); pair_df['p_holm'] = holm(pair_df['p_raw'])
print("\nSummary:")
print(pair_df.drop(columns='p_raw').round(4).to_string(index=False))


=== Ankylosing Spondylitis  vs  Rheumatoid Arthritis  (confused 551 times) ===
key tests: Anti-CCP (AUC 0.22), RF (AUC 0.24), HLA-B27 (AUC 0.75)
  all 3 measured  patients= 2192  confused=0.109
  1 missing       patients= 1470  confused=0.158
  2-3 missing     patients=  318  confused=0.252

=== Normal  vs  Sjögren's Syndrome  (confused 334 times) ===
key tests: Anti-CCP (AUC 0.24), Anti-La (AUC 0.25), RF (AUC 0.25)
  all 3 measured  patients= 1357  confused=0.093
  1 missing       patients= 1098  confused=0.135
  2-3 missing     patients=  310  confused=0.194

=== Ankylosing Spondylitis  vs  Psoriatic Arthritis  (confused 330 times) ===
key tests: ESR (AUC 0.14), Anti-CCP (AUC 0.75), RF (AUC 0.75)
  all 3 measured  patients= 1891  confused=0.084
  1 missing       patients= 1043  confused=0.129
  2-3 missing     patients=  194  confused=0.186

=== Ankylosing Spondylitis  vs  Reactive Arthritis  (confused 134 times) ===
key tests: ESR (AUC 0.93), CRP (AUC 0.84), Anti-dsDNA (AUC 0.48)
 

## 6. Are the errors low-confidence? (deferral / abstention)

**What this does:** if the model *knows* when it is unsure, a clinical system could defer those patients to a specialist. We rank patients by the model's top-class probability ("confidence"), and show the accuracy and macro-F1 on the most confident 100%, 90%, …, 50% of patients, plus the average confidence of correct vs wrong predictions.

**How to read it:** a steep rise in accuracy as coverage drops means errors are concentrated in low-confidence cases (deferral works). A flat curve means the model is confidently wrong as often as unsure — consistent with hidden overlap. Note: the probabilities are calibrated moderately well (ECE ≈ 0.015, notebook 07), which is what makes this analysis meaningful.

In [9]:
conf = P.max(1)
print(f"mean confidence: correct {conf[err == 0].mean():.3f} | wrong {conf[err == 1].mean():.3f}\n")
order = np.argsort(-conf)
rows = []
for cov in (1.0, 0.9, 0.8, 0.7, 0.6, 0.5):
    keep = order[:int(round(cov * len(y)))]
    rows.append({'coverage': cov, 'patients': len(keep), 'confidence_cutoff': conf[keep].min(),
                 'accuracy': 1 - err[keep].mean(), 'macro_f1': macro_f1_fast(y[keep], pred[keep], K),
                 'errors_removed_%': 100 * (1 - err[keep].sum() / err.sum())})
risk_cov = pd.DataFrame(rows)
print(risk_cov.round(4).to_string(index=False))

mean confidence: correct 0.872 | wrong 0.620

 coverage  patients  confidence_cutoff  accuracy  macro_f1  errors_removed_%
      1.0      9668             0.2633    0.8390    0.8254            0.0000
      0.9      8701             0.5441    0.8742    0.8574           29.6724
      0.8      7734             0.6195    0.9113    0.8929           55.9409
      0.7      6768             0.7038    0.9484    0.9304           77.5851
      0.6      5801             0.8336    0.9836    0.9674           93.8985
      0.5      4834             0.9308    0.9950    0.9831           98.4586


## 7. Save and summarise

**What this does:** writes the tables to `../results/` for the paper.

In [10]:
by_missing.to_csv('../results/err_by_n_missing.csv', index=False)
perm.drop(columns='p_raw').to_csv('../results/err_per_test.csv', index=False)
pair_df.drop(columns='p_raw').to_csv('../results/err_pairs.csv', index=False)
risk_cov.to_csv('../results/err_risk_coverage.csv', index=False)
pd.DataFrame([{'odds_ratio_per_missing_test': or_hat, 'ci_low': or_lo, 'ci_high': or_hi, 'acc_overall': acc_all,
               'acc_le1_missing': acc_low, 'verdict': verdict}]).to_csv('../results/err_verdict.csv', index=False)
print(sorted(f for f in os.listdir('../results') if f.startswith('err_')))
print("\nVERDICT:", verdict)

['err_by_n_missing.csv', 'err_pairs.csv', 'err_per_test.csv', 'err_risk_coverage.csv', 'err_verdict.csv']

VERDICT: HYPOTHESIS M SUPPORTED: missing data is a main cause of the ceiling


## Summary — what this notebook establishes

* Whether the performance ceiling (≈ 0.825 macro-F1) is better explained by missing tests or by genuine class overlap (section 3c, rule fixed in advance).
* Which missing tests cost the most accuracy and whether that matches the permutation importances of notebook 06 (section 4).
* Whether the confused disease pairs are confused even when their most discriminating tests are measured (section 5).
* Whether low-confidence flagging could catch the model's errors (section 6).